<center>
    <p style="text-align:center">
        <img alt="arize ax logo" src="https://storage.googleapis.com/arize-assets/arize-logo-white.jpg" width="100"/>
        <br>
        <br>
        <a href="https://arize.com/docs/ax/">Docs</a>
        |
        <a href="https://github.com/Arize-ai">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-2w57bhem8-hq24MB6u7yE_ZF_ilOYSBw#/shared-invite/email">Community</a>
    </p>
</center>
<h1 align="center">Aligning Evals (LLM Judge)</h1>

In this tutorial, we'll run a Mastra agent and build a custom evaluator for it. The goal is to understand the workflow for creating evaluators that align with your specific use case. We'll be working in TypeScript.

This tutorial also introduces a human-in-the-loop step for writing annotations, which are used to benchmark and measure agent performance.

Run `deno install` then `jupyter notebook` to get this notebook up and running. This notebook assumes you have already run the Mastra agent provided and generated traces in Arize AX.

## Setup

Let's start by importing the necessary packages.

In [5]:
import { listSpans, createDataset, listDatasetExamples, createExperiment } from "npm:@arizeai/ax-client@latest";
import OpenAI from "npm:openai@latest";

Set up your OpenAI API key.

In [6]:
const arizeApiKey = Deno.env.get("ARIZE_API_KEY") || prompt("Enter your Arize API key:");
const arizeSpaceId = Deno.env.get("ARIZE_SPACE_ID") || prompt("Enter your Arize Space ID or name:");
const openaiApiKey = Deno.env.get("OPENAI_API_KEY") || prompt("Enter your OpenAI API key:");

if (!arizeApiKey || !arizeSpaceId || !openaiApiKey) {
  console.error('Please enter all required credentials to continue');
  Deno.exit(1);
}

Deno.env.set("ARIZE_API_KEY", arizeApiKey);
const openai = new OpenAI({ apiKey: openaiApiKey });
const space = arizeSpaceId;

> **Note:** This notebook connects to Arize AX using your API key. Make sure you have already run the Mastra agent and that traces are visible in your Arize AX project before continuing.

In [7]:
console.log('Connected to Arize AX. Open your project at https://app.arize.com');

Connected to Arize AX. Open your project at https://app.arize.com


## Creating a Dataset

Grab the Mastra agent traces from Arize AX and format them into dataset examples. In this example, we'll extract the user query, the tool calls, and the agent's final response. Once formatted, we'll upload this dataset back into Arize AX for evaluation.

In [8]:
const { data: spans } = await listSpans({
  project: "mastra-orchestrator-workflow",
  space,
  limit: 500,
});

console.log("Total spans fetched:", spans?.length || 0);

Total spans fetched: 344


In [9]:
// Group spans by trace ID
function groupSpansByTraceId(spans: any[]) {
  const traceGroups: { [traceId: string]: any[] } = {};
  
  spans.forEach(span => {
    const traceId = span.context?.traceId || 'unknown';
    
    if (!traceGroups[traceId]) {
      traceGroups[traceId] = [];
    }
    
    traceGroups[traceId].push(span);
  });
  
  return traceGroups;
}

// Group the spans
const groupedSpans = groupSpansByTraceId(spans);
const traceIds = Object.keys(groupedSpans);

console.log(`\n📊 Found ${traceIds.length} unique traces`);
console.log("Trace IDs:", traceIds.slice(0, 5));


📊 Found 19 unique traces
Trace IDs: [
  "7282842e41d3aa7f409920e70a574a16",
  "b991c72021ccfdba7188d3d3a45fbbe8",
  "9b1705b4eb2c30953293208569ccab60",
  "dc591ea2ea3084f71d7a1a5623eb9da9",
  "1898214471703437bc2741ab483db30a"
]


In [10]:
// Analyze each trace and show span grouping
const traceAnalysis = traceIds.map(traceId => {
  const spans = groupedSpans[traceId];
  const sortedSpans = spans.sort((a: any, b: any) =>
    (a.startTime?.getTime() || 0) - (b.startTime?.getTime() || 0)
  );

  return {
    traceId,
    spanCount: spans.length,
    startTime: sortedSpans[0]?.startTime?.toISOString() || 'unknown',
    spans: sortedSpans.map((span: any) => ({
      spanId: span.context?.spanId,
      name: span.name,
      attributes: span.attributes || {}
    }))
  };
});

console.log("\n📋 Grouped Spans Structure:");
console.log("Total traces:", traceIds.length);

const datasetExamples = traceAnalysis.map(trace => {
  const userQuery = trace.spans[0]?.attributes?.['input.value'] || 'User query not found';
  const agentResponse = trace.spans[0]?.attributes?.['output.value'] || 'Agent response not found';
  const aiToolCallSpans = trace.spans.filter((span: any) => span.name === 'ai.toolCall');

  return {
    input: { userQuery },
    output: {
      agentResponse,
      aiToolCallCount: aiToolCallSpans.length,
      aiToolCallSpans: aiToolCallSpans.map((span: any) => ({
        spanId: span.spanId,
        name: span.name,
        attributes: span.attributes
      }))
    },
    metadata: {
      traceId: trace.traceId,
      source: 'mastra-orchestrator-workflow',
      timestamp: trace.startTime
    }
  };
});

console.log(`Prepared ${datasetExamples.length} examples for dataset`);

const dataset = await createDataset({
  name: `mastra-orchestrator-traces-${Date.now()}`,
  space,
  examples: datasetExamples,
});

console.log("Dataset created:", dataset.id);


📋 Grouped Spans Structure:
Total traces: 19
Prepared 19 examples for dataset
Dataset created: RGF0YXNldDozNDgwMjg6UUtXVA==


# Annotations

Next, we need human annotations to serve as ground truth for evaluation. To do this, we'll add an annotation field in the metadata of each dataset example. This way, every example includes a reference label that our evaluator outputs can be compared against.

In this example, we'll evaluate how well the agent's final response aligns with the tool calls and their outputs. We'll use three labels for evaluation: aligned, partially_aligned, and misaligned.

You can adapt this setup to other evaluation criteria as needed.

Open the dataset in Arize AX and annotate a sample of examples. Store each annotation as the `metadata.annotation` field on the example — set it to `aligned`, `partially_aligned`, or `misaligned`.

# LLM Judge Improvement Cycle

Now we'll start with a basic evaluation prompt and improve it iteratively. The workflow looks like this:

**Run the evaluator --> Inspect the outputs and experiment results --> Update the evaluation prompt based on what's lacking --> Repeat until performance improves**

We'll use Arize AX experiments to identify weaknesses in the evaluator, review explanations, and track performance changes over time.

In this notebook, we'll go through two improvement cycles, but you can extend this process with more iterations to fine-tune the evaluator further.

In [11]:
const evalPromptTemplateV1 = `
You are evaluating whether the agent's final response matches the tool outputs.

DATA:
- Query: {{query}}
- Tool Outputs & Response: {{data}}

Choose one label:
- "aligned"
- "partially_aligned"
- "misaligned"

Output only the label.
`;

In [12]:
async function runLlmJudge(
  query: string,
  data: string,
  promptTemplate: string
): Promise<{ label: string }> {
  const prompt = promptTemplate
    .replace("{{query}}", query)
    .replace("{{data}}", data);

  const response = await openai.chat.completions.create({
    model: "gpt-4o",
    messages: [{ role: "user", content: prompt }],
    temperature: 0,
  });

  const label =
    response.choices[0].message.content
      ?.trim()
      .toLowerCase()
      .replace(/['"]/g, "") || "unknown";

  return { label };
}

In [13]:
const examples = await listDatasetExamples({ dataset: dataset.id, space });

const experimentRunsV1 = await Promise.all(
  examples.map(async (example) => {
    const ex = example as any;
    const query = ex.input?.userQuery || "";
    const agentData = JSON.stringify(ex.output, null, 2);
    const annotation = ex.annotation?.Alignment?.label;

    const { label } = await runLlmJudge(query, agentData, evalPromptTemplateV1);
    const isMatch = annotation === label;

    console.log({ exampleId: example.id, query, label, annotation, isMatch });

    return {
      exampleId: example.id,
      output: JSON.stringify({
        evalLabel: label,
        humanAnnotation: annotation,
        matchesAnnotation: isMatch,
        matchScore: isMatch ? 1.0 : 0.0,
      }),
    };
  })
);

{
  exampleId: "2207cc1b-98ba-4b2e-a273-dfbc44d607a5",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "de05675b-6a33-43f3-99e7-c02170c492ed",
  query: "",
  label: "misaligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "3d367caa-baa7-46d7-b9b4-99750c12f63f",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "e2dfad6a-ef88-4e9f-ba78-dbd260baee4b",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "c8ef542f-b39b-462c-a287-8ca65cd80d85",
  query: "",
  label: "misaligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "7d9c2b12-5238-4e75-882d-90174a874f7a",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "006b2dcf-0166-43dc-bb4f-c2eca86ed68f",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "155f6724-1b3c-481d-80ba-eb70ff28cb88",
  query: "",
  

In [14]:
const experimentV1 = await createExperiment({
  experimentName: "evalTemplateV1",
  dataset: dataset.id,
  space,
  experimentRuns: experimentRunsV1,
});

console.log("Experiment created:", experimentV1.id);
console.log("View results in the Experiments tab at https://app.arize.com");

Experiment created: RXhwZXJpbWVudDo5NTc3NTptR1M2
View results in the Experiments tab at https://app.arize.com


In [15]:
const evalPromptTemplateV2 = `
You are evaluating how well an agent's FINAL RESPONSE aligns with the TOOL OUTPUTS it used.

You will be given:
- The original user query
- The agent’s final response
- The tool outputs produced by the agent

QUERY:
{{query}}

TOOL + RESPONSE DATA:
{{data}}

Choose exactly ONE label:

- "aligned" → The final response is fully supported by the tool outputs.
  * Every piece of information in the response can be traced back to the tool calls.
  * There are no additions, fabrications, or contradictions.

- "partially_aligned" → The final response mixes correct tool-based information with extra or inconsistent details.
  * Some information in the response comes from tool outputs, but other parts are missing, fabricated, or inconsistent.
  * The response is only partially grounded in the tool calls.

- "misaligned" → The final response ignores, contradicts, or invents information unrelated to the tool outputs.
  * The tool outputs do not support the response at all, or the response is in direct conflict with them.

Guidelines:
- Focus strictly on whether the content in the final response is supported by the tool outputs.
- Do not reward fluent language or style; only check alignment.
- Provide a short explanation justifying the label.

Your output must contain only one of these labels:
aligned, partially_aligned, or misaligned.
`;

In [16]:
const experimentRunsV2 = await Promise.all(
  examples.map(async (example) => {
    const ex = example as any;
    const query = ex.input?.userQuery || "";
    const agentData = JSON.stringify(ex.output, null, 2);
    const annotation = ex.annotation?.Alignment?.label;

    const { label } = await runLlmJudge(query, agentData, evalPromptTemplateV2);
    const isMatch = annotation === label;

    console.log({ exampleId: example.id, query, label, annotation, isMatch });

    return {
      exampleId: example.id,
      output: JSON.stringify({
        evalLabel: label,
        humanAnnotation: annotation,
        matchesAnnotation: isMatch,
        matchScore: isMatch ? 1.0 : 0.0,
      }),
    };
  })
);

{
  exampleId: "1d1a5e0e-ce71-4bf7-a96b-6d2899254a69",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "2207cc1b-98ba-4b2e-a273-dfbc44d607a5",
  query: "",
  label: "misaligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "006b2dcf-0166-43dc-bb4f-c2eca86ed68f",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "fadf9a55-a2c3-4601-8081-e2077119dec0",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "18ab3294-7ac0-458b-9024-0a6b0268b74b",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "e0f6440c-97e0-4138-b5b1-3fb51c1b468b",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "a2e26f5a-8a39-4b1f-80ea-514e1627ce60",
  query: "",
  label: "aligned",
  annotation: undefined,
  isMatch: false
}
{
  exampleId: "c8ef542f-b39b-462c-a287-8ca65cd80d85",
  query: "",
  lab

In [17]:
const experimentV2 = await createExperiment({
  experimentName: "evalTemplateV2",
  dataset: dataset.id,
  space,
  experimentRuns: experimentRunsV2,
});

console.log("Experiment created:", experimentV2.id);
console.log("View results in the Experiments tab at https://app.arize.com");

Experiment created: RXhwZXJpbWVudDo5NTc3NjpzYmNu
View results in the Experiments tab at https://app.arize.com
